# CS 238 Final Project — Cloud CUDA GPU Training & Zero-Shot Predictive Evaluation

**Real-Time Neurally-Updated Dynamic Bayesian Network (`STGraphTransformerDBN`) for City-Scale Autonomous Vehicle Swarm Coordination**

This self-contained notebook executes full unrolled BPTT training (`bf16` mixed precision on CUDA GPU), Boundary Lock B `model.eval()` validation selection + post-hoc L-BFGS temperature calibration on `VAL_SEEDS` (`2000..2009`), and held-out predictive evaluation (`ROC-AUC`, `PR-AUC`, `Brier`, `Log-Loss`, `10-bin ECE`) on `TEST_SEEDS` (`3000..3014`) across:
- **Held-Out ID**: `grid3x3` (`nominal`, `rush_asymmetric`, `noisy_dropout`)
- **Zero-Shot OOD Topology + Mid-Episode Incident**: `grid4x4_incident`
- **Zero-Shot Real-World OSM Urban Network**: `osm_palo_alto_rush` (Downtown Palo Alto)

Trains **both** `sota_local` ($d=128, L=4, T_{\text{seq}}=8$) and `sota_cloud_xl` ($d=256, L=8, T_{\text{seq}}=16$) presets and packages calibrated portable FP32 CPU checkpoints to `/content/drive/MyDrive/CS238_Swarm_Colab/colab_trained_models.tar.gz`.

In [ ]:
# 1. Fail-Loud CUDA GPU & BFloat16 Hardware Verification
import os
import platform
import sys
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "FAIL-LOUD: torch.cuda.is_available() is False! "
        "Switch Colab Runtime -> Change runtime type -> Hardware accelerator: GPU (A100 / L4 / T4)."
    )
if not torch.cuda.is_bf16_supported():
    raise RuntimeError(
        "FAIL-LOUD: Active CUDA device does not support bfloat16 autocast required by --precision bf16."
    )

props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / (1024 ** 3)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("=== CUDA GPU VERIFIED ===")
print(f"Python      : {sys.version.split()[0]} ({platform.platform()})")
print(f"PyTorch     : {torch.__version__} (CUDA {torch.version.cuda})")
print(f"GPU Device  : {torch.cuda.get_device_name(0)} (CC {props.major}.{props.minor})")
print(f"Total VRAM  : {vram_gb:.2f} GiB | bf16 supported: {torch.cuda.is_bf16_supported()}")
print(f"CPU Cores   : {os.cpu_count()}")

In [ ]:
# 2. Mount Google Drive & Verify Staged Rollout Archive
import hashlib
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_STAGE_DIR = Path("/content/drive/MyDrive/CS238_Swarm_Colab")
ROLLOUT_ARCHIVE = DRIVE_STAGE_DIR / "colab_rollouts.tar.gz"

if not DRIVE_STAGE_DIR.is_dir():
    raise FileNotFoundError(f"FAIL-LOUD: Missing Google Drive staging folder: {DRIVE_STAGE_DIR}")
if not ROLLOUT_ARCHIVE.is_file():
    present = [p.name for p in sorted(DRIVE_STAGE_DIR.iterdir())]
    raise FileNotFoundError(
        f"FAIL-LOUD: Missing rollout archive {ROLLOUT_ARCHIVE}. Present in folder: {present}"
    )

h = hashlib.sha256()
with ROLLOUT_ARCHIVE.open("rb") as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
        h.update(chunk)
archive_sha256 = h.hexdigest()
archive_mb = ROLLOUT_ARCHIVE.stat().st_size / (1024 ** 2)
print(f"Verified rollout archive: {ROLLOUT_ARCHIVE} ({archive_mb:.2f} MiB, sha256={archive_sha256[:16]}...)")

In [ ]:
# 3. Clone Repository (main) & Install Package Without Clobbering Colab CUDA PyTorch
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/zubadestroyer1/bn-vehicle-swarm-coordination.git"
REPO_DIR = Path("/content/bn-vehicle-swarm-coordination")

def run_checked(cmd: list[str], cwd: Path | None = None) -> str:
    print(">>>", " ".join(str(c) for c in cmd), flush=True)
    proc = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout, end="")
    if proc.returncode != 0:
        if proc.stderr:
            print(proc.stderr, file=sys.stderr)
        raise RuntimeError(f"Command failed (exit {proc.returncode}): {' '.join(cmd)}")
    return proc.stdout.strip()

if not REPO_DIR.exists():
    run_checked(["git", "clone", "--branch", "main", REPO_URL, str(REPO_DIR)])
else:
    run_checked(["git", "fetch", "origin"], cwd=REPO_DIR)
    run_checked(["git", "checkout", "main"], cwd=REPO_DIR)
    run_checked(["git", "reset", "--hard", "origin/main"], cwd=REPO_DIR)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

git_sha = run_checked(["git", "rev-parse", "HEAD"], cwd=REPO_DIR)
print(f"Checked out commit: {git_sha}")

# Install SUMO 1.28.0 wheels + scientific deps while preserving Colab's active CUDA torch build
run_checked([
    sys.executable, "-m", "pip", "install", "-q",
    "eclipse-sumo==1.28.0", "sumo-data==1.28.0", "libsumo==1.28.0",
    "traci==1.28.0", "sumolib==1.28.0",
    "scipy>=1.11", "numpy>=2.0,<3", "matplotlib>=3.8,<4", "pillow>=10",
], cwd=REPO_DIR)
run_checked([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], cwd=REPO_DIR)

run_checked([sys.executable, "-m", "swarm", "--version"], cwd=REPO_DIR)
assert torch.cuda.is_available(), "CUDA availability lost after pip install!"
print("Editable swarm installation verified cleanly.")

In [ ]:
# 4. Extract Rollout Shards & Verify Zero-Leakage Manifests + SHA-256 Hashes
import shutil
import tarfile
from swarm.pipelines.rollout_collector import load_rollout_shards

rollouts_root = REPO_DIR / "outputs" / "rollouts"
if rollouts_root.exists():
    shutil.rmtree(rollouts_root)

with tarfile.open(ROLLOUT_ARCHIVE, "r:gz") as tf:
    tf.extractall(path=REPO_DIR)

expected_splits = {
    "train": (rollouts_root / "train", "train"),
    "val": (rollouts_root / "val", "val"),
    "test_id_grid3x3": (rollouts_root / "test_id_grid3x3", None),
    "test_ood_grid4x4_incident": (rollouts_root / "test_ood_grid4x4_incident", None),
    "test_ood_palo_alto": (rollouts_root / "test_ood_palo_alto", None),
}

for label, (split_dir, exp_split) in expected_splits.items():
    manifest, shards = load_rollout_shards(split_dir, expected_split=exp_split)
    tot_edges = sum(int(r["total_candidate_edges"]) for r in manifest["shards"])
    tot_nodes = sum(int(r["total_vehicle_nodes"]) for r in manifest["shards"])
    print(
        f"[{label:26s}] episodes={len(shards):2d} | steps={manifest['num_steps_total']:5d} | "
        f"nodes={tot_nodes:7d} | cand_edges={tot_edges:8d} | split={manifest['split']}"
    )

In [ ]:
# 5. Train & Calibrate Preset 1: sota_local (d=128, L=4, T_seq=8) + Held-Out Predictive Eval
import json
import time
from swarm.dbn.updater import load_dbn_checkpoint
from swarm.viz import plot_training_and_graph_diagnostics

local_dir = REPO_DIR / "models" / "sota_local_gpu"
local_dir.mkdir(parents=True, exist_ok=True)

t0 = time.perf_counter()
run_checked([
    sys.executable, "-m", "swarm",
    "--error-report-dir", str(local_dir),
    "train",
    "--train-dir", "outputs/rollouts/train",
    "--val-dir", "outputs/rollouts/val",
    "--output-checkpoint", str(local_dir / "neural_dbn.pt"),
    "--report-path", str(local_dir / "training_report.json"),
    "--preset", "sota_local",
    "--device", "cuda",
    "--precision", "bf16",
    "--epochs", "40",
    "--batch-size", "16",
    "--lr", "3e-4",
    "--weight-decay", "1e-2",
    "--patience", "8",
    "--seed", "2026",
], cwd=REPO_DIR)
print(f"sota_local training finished in {time.perf_counter() - t0:.1f}s")

run_checked([
    sys.executable, "-m", "swarm",
    "--error-report-dir", str(local_dir),
    "eval-predictive",
    "--checkpoint", str(local_dir / "neural_dbn.pt"),
    "--named-split",
    "test_id_grid3x3=outputs/rollouts/test_id_grid3x3",
    "test_ood_grid4x4_incident=outputs/rollouts/test_ood_grid4x4_incident",
    "test_ood_palo_alto=outputs/rollouts/test_ood_palo_alto",
    "--output", str(local_dir / "predictive_report.json"),
    "--plot-output", str(local_dir / "calibration_roc_pr.png"),
    "--device", "cuda",
], cwd=REPO_DIR)

local_train_rep = json.loads((local_dir / "training_report.json").read_text(encoding="utf-8"))
plot_training_and_graph_diagnostics(local_train_rep, local_dir / "training_diagnostics.png")

# Verify portable CPU reload (weights_only=True + SHA-256 + calibrated temperatures)
local_bundle = load_dbn_checkpoint(local_dir / "neural_dbn.pt", require_calibrated=True, device="cpu")
local_pred_rep = json.loads((local_dir / "predictive_report.json").read_text(encoding="utf-8"))
for sp_name, sp_metrics in local_pred_rep["splits"].items():
    e_m = sp_metrics["edge"]
    d_m = sp_metrics["delay_post"]
    print(
        f"[sota_local | {sp_name:26s}] Edge ROC-AUC={e_m.get('roc_auc', float('nan')):.4f} "
        f"ECE={e_m['ece_pre']:.4f}->{e_m['ece_post']:.4f} | "
        f"Delay ROC-AUC={d_m.get('roc_auc', float('nan')):.4f} ECE={d_m['ece_post']:.4f}"
    )

In [ ]:
# 6. Train & Calibrate Preset 2: sota_cloud_xl (d=256, L=8, T_seq=16) + Held-Out Predictive Eval
xl_dir = REPO_DIR / "models" / "sota_cloud_xl_gpu"
xl_dir.mkdir(parents=True, exist_ok=True)
torch.cuda.empty_cache()

t0 = time.perf_counter()
run_checked([
    sys.executable, "-m", "swarm",
    "--error-report-dir", str(xl_dir),
    "train",
    "--train-dir", "outputs/rollouts/train",
    "--val-dir", "outputs/rollouts/val",
    "--output-checkpoint", str(xl_dir / "neural_dbn.pt"),
    "--report-path", str(xl_dir / "training_report.json"),
    "--preset", "sota_cloud_xl",
    "--device", "cuda",
    "--precision", "bf16",
    "--epochs", "40",
    "--batch-size", "12",
    "--lr", "2.5e-4",
    "--weight-decay", "1e-2",
    "--patience", "8",
    "--seed", "2026",
], cwd=REPO_DIR)
print(f"sota_cloud_xl training finished in {time.perf_counter() - t0:.1f}s")

run_checked([
    sys.executable, "-m", "swarm",
    "--error-report-dir", str(xl_dir),
    "eval-predictive",
    "--checkpoint", str(xl_dir / "neural_dbn.pt"),
    "--named-split",
    "test_id_grid3x3=outputs/rollouts/test_id_grid3x3",
    "test_ood_grid4x4_incident=outputs/rollouts/test_ood_grid4x4_incident",
    "test_ood_palo_alto=outputs/rollouts/test_ood_palo_alto",
    "--output", str(xl_dir / "predictive_report.json"),
    "--plot-output", str(xl_dir / "calibration_roc_pr.png"),
    "--device", "cuda",
], cwd=REPO_DIR)

xl_train_rep = json.loads((xl_dir / "training_report.json").read_text(encoding="utf-8"))
plot_training_and_graph_diagnostics(xl_train_rep, xl_dir / "training_diagnostics.png")

# Verify portable CPU reload (weights_only=True + SHA-256 + calibrated temperatures)
xl_bundle = load_dbn_checkpoint(xl_dir / "neural_dbn.pt", require_calibrated=True, device="cpu")
xl_pred_rep = json.loads((xl_dir / "predictive_report.json").read_text(encoding="utf-8"))
for sp_name, sp_metrics in xl_pred_rep["splits"].items():
    e_m = sp_metrics["edge"]
    d_m = sp_metrics["delay_post"]
    print(
        f"[sota_cloud_xl | {sp_name:26s}] Edge ROC-AUC={e_m.get('roc_auc', float('nan')):.4f} "
        f"ECE={e_m['ece_pre']:.4f}->{e_m['ece_post']:.4f} | "
        f"Delay ROC-AUC={d_m.get('roc_auc', float('nan')):.4f} ECE={d_m['ece_post']:.4f}"
    )

In [ ]:
# 7. Package Both Calibrated Checkpoints & Reports to Google Drive & Flush Cleanly
from datetime import datetime, timezone

required_files = [
    "models/sota_local_gpu/neural_dbn.pt",
    "models/sota_local_gpu/neural_dbn.pt.sha256.json",
    "models/sota_local_gpu/checkpoint_meta.json",
    "models/sota_local_gpu/training_report.json",
    "models/sota_local_gpu/predictive_report.json",
    "models/sota_local_gpu/calibration_roc_pr.png",
    "models/sota_local_gpu/training_diagnostics.png",
    "models/sota_cloud_xl_gpu/neural_dbn.pt",
    "models/sota_cloud_xl_gpu/neural_dbn.pt.sha256.json",
    "models/sota_cloud_xl_gpu/checkpoint_meta.json",
    "models/sota_cloud_xl_gpu/training_report.json",
    "models/sota_cloud_xl_gpu/predictive_report.json",
    "models/sota_cloud_xl_gpu/calibration_roc_pr.png",
    "models/sota_cloud_xl_gpu/training_diagnostics.png",
]

for rel in required_files:
    p = REPO_DIR / rel
    if not p.is_file() or p.stat().st_size == 0:
        raise FileNotFoundError(f"FAIL-LOUD: Missing or empty artifact before Drive packaging: {p}")

local_tar = REPO_DIR / "colab_trained_models.tar.gz"
with tarfile.open(local_tar, "w:gz") as tf:
    tf.add(REPO_DIR / "models" / "sota_local_gpu", arcname="models/sota_local_gpu")
    tf.add(REPO_DIR / "models" / "sota_cloud_xl_gpu", arcname="models/sota_cloud_xl_gpu")

tar_sha = hashlib.sha256(local_tar.read_bytes()).hexdigest()
drive_tar = DRIVE_STAGE_DIR / "colab_trained_models.tar.gz"
drive_tmp = DRIVE_STAGE_DIR / ".colab_trained_models.tar.gz.tmp"
shutil.copy2(local_tar, drive_tmp)
os.replace(drive_tmp, drive_tar)

summary_manifest = {
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "git_sha": git_sha,
    "gpu_name": torch.cuda.get_device_name(0),
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "rollout_archive_sha256": archive_sha256,
    "trained_models_archive": drive_tar.name,
    "trained_models_archive_sha256": tar_sha,
    "trained_models_archive_bytes": drive_tar.stat().st_size,
    "sota_local": {
        "checkpoint_sha256": local_bundle.checkpoint_sha256,
        "best_epoch": local_train_rep["best_epoch"],
        "best_val_loss": local_train_rep["best_val_loss"],
        "temperatures": local_train_rep["temperatures"],
    },
    "sota_cloud_xl": {
        "checkpoint_sha256": xl_bundle.checkpoint_sha256,
        "best_epoch": xl_train_rep["best_epoch"],
        "best_val_loss": xl_train_rep["best_val_loss"],
        "temperatures": xl_train_rep["temperatures"],
    },
}
manifest_out = DRIVE_STAGE_DIR / "colab_run_manifest.json"
manifest_out.write_text(json.dumps(summary_manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")

# Verify archive integrity from Drive path before flushing
with tarfile.open(drive_tar, "r:gz") as tf_check:
    names = tf_check.getnames()
    assert "models/sota_local_gpu/neural_dbn.pt" in names
    assert "models/sota_cloud_xl_gpu/neural_dbn.pt" in names

os.sync()
drive.flush_and_unmount()
print(f"Saved & synced {drive_tar} ({local_tar.stat().st_size / (1024 ** 2):.2f} MiB, sha256={tar_sha[:16]}...)")
print("COLAB_GPU_TRAINING_COMPLETE_OK")